# GraphRAG vs Vector RAG
## Build a Knowledge Graph Agent with Neo4j

A beginner-friendly walkthrough. You will build one small prototype from start to finish, then compare it with ordinary vector search on the same data.

**What you will learn**

1. What GraphRAG is, in plain English, before any code
2. How to build a **vector RAG** baseline with citations
3. How to design a simple **ontology** (the types of things in your graph)
4. How to extract entities, **merge duplicate names**, and load them into **Neo4j**
5. How to query the graph with **Cypher** and do **hybrid retrieval**
6. How to generate **grounded answers** that cite their sources
7. How to **evaluate** GraphRAG against vector RAG with real measured numbers
8. When a knowledge graph is **overkill**

You only need a free Colab runtime, an Anthropic API key in Colab Secrets, and about 15 minutes.

The code lives in this repo: `https://github.com/davolu/graphrag-vs-vector-rag`


## 1. Setup

We install a few libraries, load the API key safely (we never print it), and start a local Neo4j database inside this runtime so you do not need a cloud account.


In [ ]:
# On Colab, install dependencies first. On a local machine, use requirements.txt.
import sys, subprocess
pkgs = ["anthropic", "neo4j", "sentence-transformers", "rapidfuzz", "numpy"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Libraries ready.")


In [ ]:
import os, sys, subprocess
from pathlib import Path

# Find the repo root (works in Colab after a clone, and in this project folder).
ROOT = Path.cwd()
if not (ROOT / "src" / "graphrag_demo.py").exists():
    repo = Path("graphrag-vs-vector-rag")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/davolu/graphrag-vs-vector-rag.git"])
    ROOT = repo.resolve()
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from graphrag_demo import load_key, get_client, MODEL
ok = load_key()
print("ANTHROPIC_API_KEY found:", ok)
print("Model:", MODEL)
print("The key itself is never printed.")
if not ok:
    raise SystemExit("Add ANTHROPIC_API_KEY to Colab Secrets (key icon) and enable notebook access.")


In [ ]:
from graphrag_demo import start_or_connect_neo4j, connect_neo4j

# Starts a local Neo4j Community instance if nothing is already running on port 7687.
# Optional: set NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD in Secrets to use Aura instead.
driver = start_or_connect_neo4j()
print("Connected to Neo4j.")


## 2. An analogy, before any retrieval code

**Vector RAG is a librarian.** You ask a question, and they bring the pages whose wording is closest to yours. That is usually enough when the answer sits in one paragraph.

**GraphRAG is a detective's pin board.** People, companies, and products are pins. Relationships are the strings between them. When a question is really about a connection — "who manages the person who signed this contract?" — following the strings beats hunting for similar sentences.

We will build both, on the same tiny company dataset, and then keep score with real numbers.


## 3. The demo world

Cedarline Analytics is a fictional B2B company. There are 12 short notes, wiki pages, and emails. Some names are messy on purpose: the same person appears as **Maya Chen** and **M. Chen**, and the company is written as **Cedarline Analytics** and **Cedar Line Inc**. That is the kind of mess a real knowledge graph has to clean up.


In [ ]:
from graphrag_demo import load_docs

docs = load_docs()
print(f"{len(docs)} documents\n")
for d in docs:
    preview = d["text"].replace("\n", " ")
    if len(preview) > 110:
        preview = preview[:110] + "..."
    print(f"{d['id']:6}  {d['title']}")
    print(f"        {preview}\n")


In [ ]:
print("Messy names you will see in the documents:")
print("  Maya Chen  vs  M. Chen")
print("  Cedarline Analytics  vs  Cedar Line Inc")
print("  BluePeak Cloud  vs  BluePeak")


## 4. A vector RAG baseline

First we do this the usual way. Each document is already short, so we treat each one as a single chunk. We turn the chunks into embeddings with a local model, retrieve the nearest chunks, and let Claude answer with citations.


In [ ]:
from graphrag_demo import chunk_docs, embed_texts, vector_search, format_vector_context, answer_with_claude

client = get_client()
chunks = chunk_docs(docs)
matrix = embed_texts([c["text"] for c in chunks])
print(f"{len(chunks)} chunks embedded, shape {tuple(matrix.shape)}")


In [ ]:
def ask_vector(question, k=4):
    hits = vector_search(question, chunks, matrix, k=k)
    print("Retrieved:")
    for h in hits:
        print(f"  {h['doc_id']}  score={h['score']:.3f}  {h['title']}")
    result = answer_with_claude(client, question, format_vector_context(hits), "vector")
    print("\nAnswer:\n" + result["answer"])
    print(f"\nCitations: {result['citations']}   latency: {result['latency_s']} s")
    return hits, result

simple_q = "Who is the CEO of Cedarline Analytics?"
simple_hits, simple_vec = ask_vector(simple_q)


In [ ]:
hard_q = "Who manages the Cedarline person who signed the Harborline contract?"
hard_hits, hard_vec = ask_vector(hard_q)


A simple fact-finding question works. The multi-hop question is harder: the contract is in one document, and who Maya reports to is in another. If vector search does not retrieve both, the model should say it does not know rather than guess. That is the gap a graph can fill.


## 5. The ontology

Before we extract anything, we decide the shape of the world. This map is the **ontology**. It tells the model what kinds of things exist, so it does not invent a new type for every sentence.

**Node types**

- `Person` — Maya Chen, Ken Adeyemi, Priya Shah, ...
- `Organization` — Cedarline Analytics, BluePeak Cloud, Harborline Retail, ...
- `Product` — PulseBoard, PulseMail
- `Document` — the source note, kept so every fact is traceable

**Relationships**

- `WORKS_AT`, `REPORTS_TO`, `LEADS`
- `HOSTS`, `RUNS_ON`
- `SIGNED`, `CUSTOMER_OF`, `FORMERLY_WORKED_AT`
- `MENTIONED_IN` — from an entity to its source document

This tutorial uses **Neo4j and Cypher**. RDF, OWL and SPARQL are a different tradition for writing down graphs. We will stay on Neo4j here.


In [ ]:
ONTOLOGY = {
    "nodes": ["Person", "Organization", "Product", "Document"],
    "relationships": [
        "WORKS_AT", "REPORTS_TO", "LEADS", "HOSTS", "RUNS_ON",
        "SIGNED", "CUSTOMER_OF", "FORMERLY_WORKED_AT", "MENTIONED_IN",
    ],
}
print("Node types:", ", ".join(ONTOLOGY["nodes"]))
print("Relationships:")
for rel in ONTOLOGY["relationships"]:
    print(" ", rel)


## 6. Ingestion and entity resolution

Claude reads each document and pulls out entities and relationships. Then we **resolve duplicates**: M. Chen and Maya Chen become one node, Cedar Line Inc and Cedarline Analytics become one node. You should see the unique-name count drop. Every fact keeps a pointer to its source document.


In [ ]:
from graphrag_demo import extract_all, resolve_entities, load_graph_clean
import json

# Reuse a previous extraction if you already ran the pipeline in this project.
saved_ent = ROOT / "run" / "results" / "entities.json"
saved_rel = ROOT / "run" / "results" / "relations.json"
saved_stats = ROOT / "run" / "results" / "summary.json"

if saved_ent.exists() and saved_rel.exists():
    entities = json.loads(saved_ent.read_text())
    relations = json.loads(saved_rel.read_text())
    er_stats = json.loads(saved_stats.read_text())["entity_resolution"]
    print("Loaded a previous extraction from run/results/ (delete that folder to re-run).")
else:
    raw_entities, raw_relations, ext_usage = extract_all(client, docs)
    entities, relations, er_stats = resolve_entities(raw_entities, raw_relations)
    print("Extract usage (tokens in/out):", ext_usage)

print("Entity resolution:", er_stats)
print()
for e in entities:
    alias = f"  aliases={e['aliases']}" if e["aliases"] else ""
    print(f"  {e['type']:14} {e['name']}{alias}")


In [ ]:
print(f"{len(relations)} relationships after cleanup. A few examples:")
for r in relations[:8]:
    print(f"  {r['source']} -[{r['rel']}]-> {r['target']}  ({r['source_doc']})")


In [ ]:
graph_counts = load_graph_clean(driver, entities, relations, docs)
print("Loaded into Neo4j:", graph_counts)


In [ ]:
# A Cypher path for the multi-hop question: who manages the person who signed the Harborline contract?
cypher = '''
MATCH (contract_signer:Person)-[:SIGNED]->(:Organization {name: 'Harborline Retail'})
MATCH (contract_signer)-[:REPORTS_TO]->(manager:Person)
RETURN DISTINCT contract_signer.name AS signer, manager.name AS manager
'''
print(cypher.strip())
print()
with driver.session() as session:
    for rec in session.run(cypher):
        print(f"Signer: {rec['signer']}")
        print(f"Manager: {rec['manager']}")


## 7. Hybrid retrieval

Retrieval now has two steps. Vector search still finds a useful starting document or entity. Then Cypher walks two hops out and brings back the connected facts. That is what lets the agent answer relationship questions.


In [ ]:
from graphrag_demo import hybrid_retrieve, format_graph_context

bundle = hybrid_retrieve(hard_q, chunks, matrix, entities, driver, k=3, hops=2)
print("Vector seeds:", bundle["seeds"])
print(f"Graph facts returned: {len(bundle['facts'])}")
print()
for f in bundle["facts"][:12]:
    print(f"  {f['source']} -[{f['rel']}]-> {f['target']}  ({f.get('source_doc')})")


## 8. Grounded answers with citations

When we ask the same multi-hop question again, Claude may only use the retrieved facts and must cite source documents. If the graph does not support a claim, it should say it does not know.


In [ ]:
from graphrag_demo import answer_with_claude

graph_result = answer_with_claude(client, hard_q, format_graph_context(bundle), "graph")
print("GraphRAG answer:\n")
print(graph_result["answer"])
print(f"\nCitations: {graph_result['citations']}   latency: {graph_result['latency_s']} s")
print("\nVector RAG said:\n")
print(hard_vec["answer"])


## 9. Evaluation against the vector baseline

We score both systems on the same eight questions, with a deterministic checker: required phrases must appear, and an "I do not know" is not counted as correct when there is a gold answer. Latency is measured with `time.perf_counter`. No estimates.


In [ ]:
from graphrag_demo import load_eval_questions, score_answer

summary_path = ROOT / "run" / "results" / "summary.json"
rows_path = ROOT / "run" / "results" / "rows.json"

if summary_path.exists() and rows_path.exists():
    summary = json.loads(summary_path.read_text())
    eval_rows = json.loads(rows_path.read_text())
    print("Using the measured run in run/results/.")
else:
    questions = load_eval_questions()
    eval_rows = []
    for item in questions:
        q = item["question"]
        vec = answer_with_claude(client, q, format_vector_context(vector_search(q, chunks, matrix, 4)), "vector")
        b = hybrid_retrieve(q, chunks, matrix, entities, driver, k=3, hops=2)
        gra = answer_with_claude(client, q, format_graph_context(b), "graph")
        eval_rows.append({"id": item["id"], "kind": item["kind"], "question": q, "gold": item["gold"],
                          "vector": score_answer(item, vec), "graphrag": score_answer(item, gra)})
    def _sum(key):
        xs = [r[key] for r in eval_rows]; n = len(xs)
        return {"n": n, "correct": sum(x["correct"] for x in xs),
                "citation_ok": sum(x["citation_ok"] for x in xs),
                "accuracy": round(sum(x["correct"] for x in xs) / n, 3),
                "citation_accuracy": round(sum(x["citation_ok"] for x in xs) / n, 3),
                "avg_latency_s": round(sum(x["latency_s"] for x in xs) / n, 3)}
    summary = {"vector": _sum("vector"), "graphrag": _sum("graphrag")}

print(f"{'ID':4} {'Kind':10} {'Vector':8} {'GraphRAG':8} {'V lat':7} {'G lat':7}")
print("-" * 52)
for r in eval_rows:
    print(f"{r['id']:4} {r['kind']:10} {str(r['vector']['correct']):8} {str(r['graphrag']['correct']):8} "
          f"{r['vector']['latency_s']:<7} {r['graphrag']['latency_s']:<7}")
print()
v, g = summary["vector"], summary["graphrag"]
print(f"Vector RAG  {v['correct']}/{v['n']} correct  ({v['accuracy']:.1%})  citations {v['citation_ok']}/{v['n']}  avg {v['avg_latency_s']} s")
print(f"GraphRAG    {g['correct']}/{g['n']} correct  ({g['accuracy']:.1%})  citations {g['citation_ok']}/{g['n']}  avg {g['avg_latency_s']} s")


## 10. When the graph is overkill

A graph is not always better. For a question that already sits in one paragraph, such as a refund policy, vector search is simpler and just as accurate. On this run it was also slightly faster on average. Choose the tool that fits the question.


In [ ]:
policy_q = "How do customers get a service credit if PulseBoard is down for more than 8 hours in a month?"
print("Question:", policy_q, "\n")
print("=== Vector RAG ===")
_, policy_vec = ask_vector(policy_q)
print("\n=== GraphRAG ===")
policy_bundle = hybrid_retrieve(policy_q, chunks, matrix, entities, driver, k=3, hops=2)
policy_gra = answer_with_claude(client, policy_q, format_graph_context(policy_bundle), "graph")
print(policy_gra["answer"])
print(f"Citations: {policy_gra['citations']}   latency: {policy_gra['latency_s']} s")


## Recap

You now have a small prototype:

1. An ontology — the types of nodes and relationships
2. Entity resolution — messy names become one node
3. Neo4j and Cypher — facts you can walk as paths
4. Hybrid retrieval — vector search to start, then graph expansion
5. Grounded answers with citations
6. A fair score against a vector baseline, using real measured numbers

On this dataset, vector RAG got **7 / 8** questions right (87.5%, average 0.86 s). GraphRAG got **8 / 8** (100%, average 0.93 s). The extra win was a multi-hop management question, where the contract and the reporting line lived in different documents. The policy question was a tie: the graph was not needed.

The notebook and data are in the GitHub repo linked in the description. Next up: **Agentic RAG**, where the agent decides *how* to search.

David Oluyale  |  Agentic AI Hub
